In [1]:
import sys
print(sys.executable)

C:\Users\pc\anaconda3\envs\voila_env\python.exe


<h2 style="font-family:Verdana; color:green;">Tutorial Step 1</h2>
<p style="font-size:16px;">Write your instructions here in a readable font and size.</p>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import ipywidgets as widgets
from IPython.display import display, clear_output
from IPython.display import display, HTML


# Input values
theta_i_deg = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90])
n = 1.5  # n2/n1

# Define equations
def TE_r(theta):
    return (np.cos(theta) - np.sqrt(n**2 - np.sin(theta)**2)) / \
           (np.cos(theta) + np.sqrt(n**2 - np.sin(theta)**2))

def TE_t(theta):
    return (2*np.cos(theta))/(np.cos(theta) + np.sqrt(n**2 - np.sin(theta)**2))

def simple_eq(theta):
    return theta + 1  # example: x + 1

# Dictionary of equations
equations = {
    "TE_r": TE_r,
    "TE_t": TE_t,
    "x + 1": simple_eq
}

# General function to create DataFrame and plot
def plot_mode(equation, theta_deg, title, color='darkblue'):
    theta_rad = np.radians(theta_deg)
    y_values = equation(theta_rad)
    
    # Create the DataFrame
    df = pd.DataFrame({
        "Theta": theta_deg,
        "Value": y_values
    })
    
    # Style the DataFrame (exactly as in your original code)
    styled = df.style.set_table_styles(
        [{'selector': 'th', 'props': [('background-color', '#f2f2f2'), 
                                      ('color', 'black'), 
                                      ('font-family', 'Serif'), 
                                      ('font-size', '16px')]}]
    ).set_properties(**{
        'text-align': 'center',
        'font-size': '16px',
        'font-family': 'Serif'
    })
    display(styled)

    # Create the scatter plot from the data
    X = df[["Theta"]].values   # input
    y = df["Value"].values     # output
    
    best_degree = 1 # stores the best polynomial degree
    best_error = float("inf") # starts at infinity so any real error is smaller
    
    # Loop to find best polynomial degree
    for d in range(1, 6):  # try degrees 1 to 5
        poly = PolynomialFeatures(degree=d)
        X_poly = poly.fit_transform(X)
        
        model = LinearRegression()
        
        # Calculate error (lower is better) using cross-validation
        error = -cross_val_score(
            model, X_poly, y,
            scoring="neg_mean_squared_error", # average of squared differences
            cv=3
        ).mean()
        
        if error < best_error:
            best_error = error
            best_degree = d

    print("Best degree:", best_degree)

    # Fit model with best degree
    poly = PolynomialFeatures(degree=best_degree)
    X_poly = poly.fit_transform(X)

    model = LinearRegression()
    model.fit(X_poly, y)

    # Drawing the smooth curve
    x_smooth = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    x_smooth_poly = poly.transform(x_smooth)
    y_smooth = model.predict(x_smooth_poly)

    # Plotting with the exact style from your original code
    plt.figure(figsize=(10, 5))
    plt.scatter(X, y, alpha=0.7, color=color, marker='.')
    plt.plot(x_smooth, y_smooth, color=color)
    plt.xlabel('Theta-deg', fontsize=12, fontfamily='Serif')
    plt.ylabel('Value', fontsize=12, fontfamily='Serif')
    plt.title(title, fontsize=16, fontfamily='Serif')
    plt.xticks(fontsize=10, fontfamily='Serif')
    plt.yticks(fontsize=10, fontfamily='Serif')
    plt.grid(True)
    plt.show()

# Interactive widget function
def interactive_plot(mode_name):
    clear_output(wait=True)
    eq_func = equations[mode_name]
    # Pick a color for each equation
    color_map = {"TE_r": "darkblue", "TE_t": "red", "x + 1": "green"}
    plot_mode(eq_func, theta_i_deg, title=mode_name, color=color_map.get(mode_name, "black"))

# Dropdown widget
dropdown = widgets.Dropdown(
    options=list(equations.keys()),
    value="TE_r",
    description="Equation:",
)

widgets.interact(interactive_plot, mode_name=dropdown)

interactive(children=(Dropdown(description='Equation:', options=('TE_r', 'TE_t', 'x + 1'), value='TE_r'), Outp…

<function __main__.interactive_plot(mode_name)>